## Setup

# CFD Data Processing - Part 4: Coefficient Graphs

This notebook creates aerodynamic coefficient plots (C_L and C_D vs AoA).
Works with or without convergence analysis results from Part 3.

Creates 4 graphs per configuration: C_L vs AoA, C_D vs AoA, Drag Polar, and Combined view.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from collections import defaultdict
import glob
import time

## Load Processed Data

In [2]:
# Automatically find the pickle file generated by Part 1
# This searches for processed_data.pkl starting from user's home directory

# Start search from a higher directory to find the Thesis folder
# Try multiple search locations to find the pickle file
search_roots = [
    r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis",  # Direct path to Thesis folder
    os.path.expanduser("~"),  # User home directory
]

pickle_files = []
for root in search_roots:
    if os.path.exists(root):
        search_pattern = os.path.join(root, "**", "processed_data.pkl")
        found = glob.glob(search_pattern, recursive=True)
        pickle_files.extend(found)
        if found:
            break  # Stop searching once we find files

# Filter to get the most recently modified pickle file
if pickle_files:
    pickle_file = max(pickle_files, key=os.path.getmtime)
    print(f"✓ Found pickle file: {pickle_file}")
    print(f"  Last modified: {time.ctime(os.path.getmtime(pickle_file))}")
else:
    print("\n❌ ERROR: No processed_data.pkl file found!")
    print("\n💡 Solution:")
    print("   Run notebook 1_data_processing.ipynb first to generate processed_data.pkl")
    raise FileNotFoundError(f"Run Part 1 first to create processed_data.pkl")

with open(pickle_file, 'rb') as f:
    data_package = pickle.load(f)

all_data = data_package['all_data']
config_info = data_package['config_info']
paths = data_package['paths']

POSITION_MAP = config_info['POSITION_MAP']
VALUE_MAPPINGS = config_info['VALUE_MAPPINGS']
TURBULENCE_ORDER = config_info['TURBULENCE_ORDER']

base_path = paths['base_path']
output_dir = paths['output_dir']

print(f"✓ Data loaded: {len(all_data)} configurations")
print(f"✓ Output directory: {output_dir}")

✓ Found pickle file: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\processed_data.pkl
  Last modified: Sun Dec  7 14:52:27 2025
✓ Data loaded: 16 configurations
✓ Output directory: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG


## User Settings

In [3]:
# User settings
num_iterations = 150  # Used if no convergence results available

# Coefficient parameters
SPAN = 0.85344
CHORD = 0.1
AIR_DENSITY = 1.225
VELOCITY = 24.38

REFERENCE_AREA = SPAN * CHORD
DYNAMIC_PRESSURE = 0.5 * AIR_DENSITY * VELOCITY**2
Q_times_A = DYNAMIC_PRESSURE * REFERENCE_AREA

# Try to load convergence results if available
convergence_pickle = os.path.join(output_dir, 'convergence_results.pkl')
if os.path.exists(convergence_pickle):
    with open(convergence_pickle, 'rb') as f:
        conv_package = pickle.load(f)
        convergence_results = conv_package['convergence_results']
        RUN_CONVERGENCE_ANALYSIS = True
        print(f"✓ Using convergence results from Part 3")
        print(f"  Configurations optimized: {len(convergence_results)}")
else:
    convergence_results = {}
    RUN_CONVERGENCE_ANALYSIS = False
    print(f"⚠ No convergence results found")
    print(f"  Using fixed iterations: {num_iterations} points")

print(f"\nQ × A = {Q_times_A:.4f}")

✓ Using convergence results from Part 3
  Configurations optimized: 16

Q × A = 31.0704


## Helper Functions

In [4]:
def extract_aoa_number(aoa_string):
    try:
        return int(aoa_string.split('_')[1])
    except:
        return 0

## Coefficient Graphs (C_L and C_D vs AoA)

In [ ]:
# ==================== CALCULATE COEFFICIENTS ====================
# Calculate C_L and C_D for each configuration

# Initialize convergence_results if not already defined (when RUN_CONVERGENCE_ANALYSIS is False)
if 'convergence_results' not in dir():
    convergence_results = {}

coefficient_data = {}

for (config, aoa), data in all_data.items():
    # Get optimized data if convergence analysis was run, otherwise use fixed iterations
    if RUN_CONVERGENCE_ANALYSIS and (config, aoa) in convergence_results:
        conv = convergence_results[(config, aoa)]
        lift_min_cov_idx = np.argmin(conv['lift']['cov'])
        drag_min_cov_idx = np.argmin(conv['drag']['cov'])
        
        optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
        optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
        optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
        
        lift_values = data['lift'][optimal_trim:]
        drag_values = data['drag'][optimal_trim:]
    else:
        # Use fixed iterations approach
        lift_values = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_values = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate mean forces
    lift_mean = np.mean(lift_values) if lift_values else 0
    drag_mean = np.mean(drag_values) if drag_values else 0
    
    # Calculate coefficients
    C_L = lift_mean / Q_times_A if Q_times_A != 0 else 0
    C_D = drag_mean / Q_times_A if Q_times_A != 0 else 0
    
    # Calculate standard deviations and COV for coefficients
    lift_std = np.std(lift_values) if lift_values else 0
    drag_std = np.std(drag_values) if drag_values else 0
    
    C_L_std = lift_std / Q_times_A if Q_times_A != 0 else 0
    C_D_std = drag_std / Q_times_A if Q_times_A != 0 else 0
    
    C_L_cov = (C_L_std / C_L * 100) if C_L != 0 else 0
    C_D_cov = (C_D_std / C_D * 100) if C_D != 0 else 0
    
    # Extract AoA number
    aoa_degrees = extract_aoa_number(aoa)
    
    # Extract config parts for filtering
    config_parts = config.split('.')
    
    # Get version number from config
    version_num = config_parts[POSITION_MAP['version']] if len(config_parts) > POSITION_MAP['version'] else 'unknown'
    
    # Store coefficient data
    coefficient_data[(config, aoa)] = {
        'config': config,
        'config_parts': config_parts,
        'version_num': version_num,
        'aoa_degrees': aoa_degrees,
        'C_L': C_L,
        'C_D': C_D,
        'C_L_std': C_L_std,
        'C_D_std': C_D_std,
        'C_L_cov': C_L_cov,
        'C_D_cov': C_D_cov,
        'turbulence_model': data['turbulence_model'],
        'geometry': data['geometry'],
        'mesh': data['mesh'],
        'grid': data['grid']
    }

print(f"\n✓ Coefficients calculated for {len(coefficient_data)} configurations")

# ==================== CREATE C_L vs AoA and C_D vs AoA GRAPHS ====================
# Create folder structure: coefficient_graphs / turbulence_model / config

# Get all unique base configurations (without AoA - first 4 parts)
unique_base_configs = set()
for (config, aoa), coeff in coefficient_data.items():
    parts = config.split('.')
    if len(parts) >= 4:
        base_config = '.'.join(parts[:4])  # e.g., "4.3.1.2"
        unique_base_configs.add(base_config)

# Group configurations by turbulence model
configs_by_turbulence = defaultdict(list)
for base_config in unique_base_configs:
    parts = base_config.split('.')
    turb_num = parts[POSITION_MAP['turbulence']] if len(parts) > POSITION_MAP['turbulence'] else 'unknown'
    turb_name = VALUE_MAPPINGS['turbulence'].get(int(turb_num) if turb_num.isdigit() else turb_num, f"Turbulence_{turb_num}")
    configs_by_turbulence[turb_name].append(base_config)

print(f"\n✓ Found {len(unique_base_configs)} unique configurations organized by turbulence model:")
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"\n  {turb_name}/")
    for cfg in sorted(configs_by_turbulence[turb_name]):
        print(f"    - {cfg}/")

# Define colors and markers for different turbulence models
model_styles = {
    'SST': {'color': '#1f77b4', 'marker': 'o', 'label': 'SST'},
    'RNG': {'color': '#ff7f0e', 'marker': 's', 'label': 'RNG k-ε'},
    'RSM': {'color': '#2ca02c', 'marker': '^', 'label': 'RSM'},
    'k-epsilon': {'color': '#d62728', 'marker': 'D', 'label': 'k-ε'}
}

# Create graphs organized by turbulence model
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"\n{'='*70}")
    print(f"TURBULENCE MODEL: {turb_name}")
    print(f"{'='*70}")
    
    for base_config in sorted(configs_by_turbulence[turb_name]):
        # Create directory: coefficient_graphs / turbulence_model / config
        config_graphs_dir = os.path.join(output_dir, "coefficient_graphs", turb_name, base_config)
        os.makedirs(config_graphs_dir, exist_ok=True)
        
        # Get descriptive info for this config
        parts = base_config.split('.')
        geometry_name = VALUE_MAPPINGS['geometry'].get(int(parts[0]) if parts[0].isdigit() else parts[0], f"Geometry_{parts[0]}")
        mesh_name = VALUE_MAPPINGS['mesh'].get(int(parts[1]) if parts[1].isdigit() else parts[1], f"Mesh_{parts[1]}")
        version_name = VALUE_MAPPINGS['version'].get(int(parts[3]) if parts[3].isdigit() else parts[3], f"Version_{parts[3]}")
        
        config_description = f"{geometry_name}, {mesh_name}, {turb_name}, {version_name}"
        
        print(f"\n  Creating graphs for {base_config}")
        print(f"    ({config_description})")
        
        # Filter coefficient data for this base configuration
        config_coeff_data = {}
        for (config, aoa), coeff in coefficient_data.items():
            config_parts = config.split('.')
            if len(config_parts) >= 4:
                this_base = '.'.join(config_parts[:4])
                if this_base == base_config:
                    config_coeff_data[(config, aoa)] = coeff
        
        if not config_coeff_data:
            print(f"    ⚠ No data found for {base_config}")
            continue
        
        # Organize data by AoA for plotting
        aoa_list = []
        C_L_list = []
        C_D_list = []
        C_L_std_list = []
        C_D_std_list = []
        
        for (config, aoa), coeff in config_coeff_data.items():
            aoa_list.append(coeff['aoa_degrees'])
            C_L_list.append(coeff['C_L'])
            C_D_list.append(coeff['C_D'])
            C_L_std_list.append(coeff['C_L_std'])
            C_D_std_list.append(coeff['C_D_std'])
        
        # Sort by AoA
        combined = list(zip(aoa_list, C_L_list, C_D_list, C_L_std_list, C_D_std_list))
        combined.sort(key=lambda x: x[0])
        
        aoa_vals = np.array([x[0] for x in combined])
        C_L_vals = np.array([x[1] for x in combined])
        C_D_vals = np.array([x[2] for x in combined])
        C_L_std_vals = np.array([x[3] for x in combined])
        C_D_std_vals = np.array([x[4] for x in combined])
        
        # Get style based on turbulence model
        style = model_styles.get(turb_name, {'color': '#1f77b4', 'marker': 'o', 'label': turb_name})
        
        # ==================== PLOT 1: C_L vs AoA ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=18, fontweight='bold')
        ax.set_title(f'Lift Coefficient vs Angle of Attack\n{base_config}', fontsize=18, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=16, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=15)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_L_vs_AoA.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 2: C_D vs AoA ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax.set_ylabel('Drag Coefficient ($C_D$)', fontsize=18, fontweight='bold')
        ax.set_title(f'Drag Coefficient vs Angle of Attack\n{base_config}', fontsize=18, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=16, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=15)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_D_vs_AoA.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 3: C_L vs C_D (Drag Polar) ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(C_D_vals, C_L_vals, xerr=C_D_std_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        # Add AoA annotations at each point
        for i, aoa in enumerate(aoa_vals):
            ax.annotate(f"{int(aoa)}°", 
                       (C_D_vals[i], C_L_vals[i]),
                       textcoords="offset points", xytext=(8, 5),
                       fontsize=14, fontweight='bold', color=style['color'])
        
        ax.set_xlabel('Drag Coefficient ($C_D$)', fontsize=18, fontweight='bold')
        ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=18, fontweight='bold')
        ax.set_title(f'Drag Polar ($C_L$ vs $C_D$)\n{base_config}', fontsize=18, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=16, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=15)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "Drag_Polar.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 4: Combined C_L and C_D vs AoA ====================
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        
        # Left plot: C_L vs AoA
        ax1.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax1.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax1.set_ylabel('Lift Coefficient ($C_L$)', fontsize=18, fontweight='bold')
        ax1.set_title(f'Lift Coefficient vs AoA\n{base_config}', fontsize=18, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.legend(fontsize=15, loc='best', framealpha=0.9)
        ax1.tick_params(labelsize=15)
        
        # Right plot: C_D vs AoA
        ax2.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax2.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax2.set_ylabel('Drag Coefficient ($C_D$)', fontsize=18, fontweight='bold')
        ax2.set_title(f'Drag Coefficient vs AoA\n{base_config}', fontsize=18, fontweight='bold')
        ax2.grid(True, alpha=0.3, linestyle='--')
        ax2.legend(fontsize=15, loc='best', framealpha=0.9)
        ax2.tick_params(labelsize=15)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_L_C_D_Combined.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"    ✓ All 4 graphs saved to: {turb_name}/{base_config}/")

print("\n" + "=" * 80)
print("COEFFICIENT GRAPHS COMPLETE")
print(f"✓ Folder structure: coefficient_graphs / turbulence_model / config")
print(f"✓ Location: {os.path.join(output_dir, 'coefficient_graphs')}")
print(f"\nFolder structure:")
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"  {turb_name}/")
    for cfg in sorted(configs_by_turbulence[turb_name]):
        print(f"    └── {cfg}/")
print("\n✓ Each config folder contains: C_L_vs_AoA.png, C_D_vs_AoA.png, Drag_Polar.png, C_L_C_D_Combined.png")


✓ Coefficients calculated for 16 configurations

✓ Found 1 unique configurations organized by turbulence model:

  Turbulence_1/
    - 3.1.1.NG/

TURBULENCE MODEL: Turbulence_1

  Creating graphs for 3.1.1.NG
    (Geometry_3, Mesh_1, Turbulence_1, Version_NG)
    ✓ All 4 graphs saved to: Turbulence_1/3.1.1.NG/

COEFFICIENT GRAPHS COMPLETE
✓ Folder structure: coefficient_graphs / turbulence_model / config
✓ Location: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\coefficient_graphs

Folder structure:
  Turbulence_1/
    └── 3.1.1.NG/

✓ Each config folder contains: C_L_vs_AoA.png, C_D_vs_AoA.png, Drag_Polar.png, C_L_C_D_Combined.png
    ✓ All 4 graphs saved to: Turbulence_1/3.1.1.NG/

COEFFICIENT GRAPHS COMPLETE
✓ Folder structure: coefficient_graphs / turbulence_model / config
✓ Location: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\coefficient_graphs

Folder structure